In [2]:
from neo4j import GraphDatabase
import re
from pathlib import Path

# Folder-based source used by extract_structured_texts_from_folder
SOURCE_TEXT_FOLDER = "dataOran/3gpp-specifications-md"

# Single Neo4j instance used only for KG relationships (triplets)
URI = "neo4j://localhost:7687"
NEO4J_AUTH = ("neo4j", "testpassword")

# Two separate vector storages for title retrieval
PARAGRAPH_VECTOR_STORE_PATH = "paragraph_title_vectors.json"
SECTION_VECTOR_STORE_PATH = "section_title_vectors.json"
EMBEDDING_MODEL = "nomic-embed-text"

In [3]:
def extract_structured_text(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()

        # Split the text into blocks separated by at least one empty line
        raw_blocks = content.split('\n\n')

        structured_data = []

        for block in raw_blocks:
            # Clean the block and split into internal lines
            lines = [line.strip() for line in block.split('\n') if line.strip()]

            if not lines:
                continue

            if len(lines) == 1:
                structured_data.append({
                    "type": "Title",
                    "text": lines[0],
                })
            else:
                structured_data.append({
                    "type": "Paragraph",
                    "text": " ".join(lines),
                })

        return structured_data

    except FileNotFoundError:
        print(f"Error: File not found -> {file_path}")
        return []

def extract_structured_texts_from_folder(folder_path):
    folder = Path(folder_path)
    txt_files = sorted(folder.rglob("*.txt"))

    if not txt_files:
        print(f"No .txt files found in folder: {folder_path}")
        return [], []

    combined_data = []
    for txt_file in txt_files:
        file_data = extract_structured_text(txt_file)
        for item in file_data:
            item["source_file"] = txt_file.name
        combined_data.extend(file_data)

    return combined_data, txt_files

# Execution (folder mode)
data, source_files = extract_structured_texts_from_folder(SOURCE_TEXT_FOLDER)

print(f"Loaded {len(source_files)} text files from: {SOURCE_TEXT_FOLDER}")
print(f"Total extracted blocks: {len(data)}")

# Displaying sample results
for item in data[:20]:
    print(f"[{item['type'].upper()}] ({item.get('source_file', 'N/A')}): {item['text'][:70]}...")

Loaded 45 text files from: dataOran/3gpp-specifications-md
Total extracted blocks: 59527
[PARAGRAPH] (23501-k00.txt): --- title: "23501-k00" ---...
[TITLE] (23501-k00.txt): # 3 Definitions and abbreviations...
[TITLE] (23501-k00.txt): ## 3.1 Definitions...
[TITLE] (23501-k00.txt): For the purposes of the present document, the terms and definitions gi...
[TITLE] (23501-k00.txt): 5G VN Group: A set of UEs using private communication for 5G LAN-type ...
[TITLE] (23501-k00.txt): 5G Access Network: An access network comprising a NG-RAN and/or non-3G...
[TITLE] (23501-k00.txt): 5G Access Stratum-based Time Distribution: A time synchronization dist...
[TITLE] (23501-k00.txt): 5G Core Network: The core network specified in the present document. I...
[TITLE] (23501-k00.txt): 5G LAN-Type Service: A service over the 5G system offering private com...
[TITLE] (23501-k00.txt): 5G LAN-Virtual Network: A virtual network over the 5G system capable o...
[TITLE] (23501-k00.txt): 5G NSWO: The 5G NSWO is t

In [7]:
import re

def stream_relationships(large_text):
    # This pattern specifically targets the Relationship Format:
    # It looks for: #Number [Subject] -> [Verb] -> [Object]
    # We use \s* to handle any inconsistent spacing around the arrows.
    rel_pattern = re.compile(r"#(\d+)\s+(.*?)\s*->\s*(.*?)\s*->\s*(.*)")

    # Splitting by newline to simulate line-by-line reading
    for line in large_text.splitlines():
        clean_line = line.strip()
        
        # We only care about lines that match our specific relationship pattern
        match = rel_pattern.search(clean_line)
        if match:
            yield {
                "id": int(match.group(1)),
                "subject": match.group(2).strip(),
                "verb": match.group(3).replace(" ",""),
                "object": match.group(4).strip()
            }


In [5]:
import ollama
import json

def self_discover_and_extract(title, paragraphs, entity_types, predicates):
    sample_text = f"Title: {title}\n\n" + "\n".join(paragraphs[:3])

    discovery_prompt = f"""Perform Named Entity Recognition (NER) and extract knowledge graph triplets from the text. NER identifies named entities of given entity types, and triple extraction identifies relationships between entities using specified predicates.

        **Entity Types:**
        {json.dumps(entity_types)}

        **Predicates:**
        {json.dumps(predicates)}

        **Text:**
        {sample_text}

        **Example Output:**
            "entities": [{{ "text": "Google", "type": "ORGANIZATION" }}],
            "triplets": [{{ "subject": "Google", "predicate": "founded_in", "object": "USA" }}]
        """

    discovery_prompt=f""" 
                        Prompt for extracting entities: Extract key entities from the given
                        text. Extracted entities are nouns, verbs, or adjectives,
                        particularly regarding sentiment. This is for an extraction
                        task, please be thorough and accurate to the reference text.

                        Prompt for extracting relations: Extract subject-predicate-object
                        triples from the assistant message. A predicate (1-3
                        words) defines the relationship between the subject and
                        object. Relationship may be fact or sentiment based on
                        assistant’s message. Subject and object are entities.
                        Entities provided are from the assistant message and
                        prior conversation history, though you may not need all of
                        them. This is for an extraction task, please be thorough,
                        accurate, and faithful to the reference text

                        In the end represent all the relationships with this format, do not add text:
                        
                        *Relationship Format:*

                        #[Number] Entity -> Predicate -> Entity

                        {sample_text}
                        """

    # Remove the redundant .format() call — the f-string already handled substitution
    discovery_response = ollama.chat(model='llama3:8b', messages=[
        {'role': 'user', 'content': discovery_prompt}
    ])
    return discovery_response['message']['content']

# Usage
# schema, triplets = self_discover_and_extract(my_title, my_paragraphs)for p in range(len(data)-1):

for p in range(len(data)-1):
    current_val =data[p]
    next_val = data[p+1]

    if current_val['type'] == 'Title' and next_val['type'] == 'Paragraph':
        my_title = current_val['text']
        my_paragraphs = [next_val['text']]

        

        entity_types = ["Person", "Organization", "Object", "Subject"]
        predicates = ["related_to", "located_in", "participated_in"]

        data_n = self_discover_and_extract(my_title, my_paragraphs, entity_types, predicates)
        extracted_triplets = list(stream_relationships(data_n))
        # 3. Print the actual parsed dictionaries
        print(f"--- Results for: {my_title} ---")
        for triplet in extracted_triplets:
            print(triplet)

--- Results for: Introduction ---
{'id': 1, 'subject': 'LLMs', 'verb': 'represent', 'object': 'computational systems'}
{'id': 2, 'subject': 'LLMs', 'verb': 'understand', 'object': 'human language'}
{'id': 3, 'subject': 'N-gram models', 'verb': 'leverage', 'object': 'Transformer architectures'}
{'id': 4, 'subject': 'GPT-3 and GPT-4', 'verb': 'leverage', 'object': 'self-attention mechanism within Transformer architectures'}
{'id': 5, 'subject': 'LLMs', 'verb': 'generate', 'object': 'coherent text from prompts'}
{'id': 6, 'subject': 'RLHF', 'verb': 'refine', 'object': 'models using human responses'}
{'id': 7, 'subject': 'Techniques (prompt engineering, question-answering, and conversational interactions)', 'verb': 'advance', 'object': 'field of natural language processing (NLP)'}
--- Results for: Stage 1: Data Preparation ---
--- Results for: 4.1     Steps Involved in Model Initialisation ---
{'id': 1, 'subject': 'Model', 'verb': 'IsInvolvedIn', 'object': 'Steps'}
{'id': 2, 'subject': 'St

In [6]:
def upload_triplets(driver, triplets, paragraph_name):
    with driver.session() as session:
        for triplet in triplets:
            rel_type = re.sub(r'[^a-zA-Z0-9_]', '', triplet['verb'].replace(" ", "_").upper())
            if not rel_type:
                rel_type = "RELATED_TO"

            # Keep relation and add paragraph context links for both entities.
            query = (
                f"MERGE (s:Entity {{name: $sub}}) "
                f"MERGE (o:Entity {{name: $obj}}) "
                f"MERGE (p:Paragraph {{name: $paragraph_name}}) "
                f"MERGE (s)-[:{rel_type}]->(o) "
                f"MERGE (s)-[:MENTIONED_IN]->(p) "
                f"MERGE (o)-[:MENTIONED_IN]->(p)"
            )
            session.run(
                query,
                sub=triplet['subject'],
                obj=triplet['object'],
                paragraph_name=paragraph_name,
            )

In [7]:
def upload_triplets_APOC(driver, triplets, paragraph_name):
    with driver.session() as session:
        for triplet in triplets:
            # Clean the verb to make it a valid Neo4j Relationship Type
            rel_type = re.sub(r'[^a-zA-Z0-9_]', '', triplet['verb'].replace(" ", "_").upper())
            if not rel_type:
                rel_type = "RELATED_TO"

            query = """
            MERGE (s:Entity {name: $sub})
            MERGE (o:Entity {name: $obj})
            MERGE (p:Paragraph {name: $paragraph_name})
            WITH s, o, p
            CALL apoc.create.relationship(s, $rel, {}, o) YIELD rel
            MERGE (s)-[:MENTIONED_IN]->(p)
            MERGE (o)-[:MENTIONED_IN]->(p)
            RETURN rel
            """
            session.run(
                query,
                sub=triplet['subject'],
                obj=triplet['object'],
                paragraph_name=paragraph_name,
                rel=rel_type,
            )

In [8]:
# Integration with Ollama loop + paragraph-context links
with GraphDatabase.driver(URI, auth=NEO4J_AUTH) as driver:
    driver.verify_connectivity()

    for p in range(len(data) - 1):
        current_val = data[p]
        next_val = data[p + 1]

        if current_val['type'] == 'Title' and next_val['type'] == 'Paragraph':
            my_title = current_val['text']
            my_paragraphs = [next_val['text']]

            entity_types = ["Person", "Organization", "Object", "Subject"]
            predicates = ["related_to", "located_in", "participated_in"]

            data_n = self_discover_and_extract(my_title, my_paragraphs, entity_types, predicates)
            triplets = list(stream_relationships(data_n))

            if triplets:
                upload_triplets(driver, triplets, my_title)
                print(f"Uploaded {len(triplets)} relationships for paragraph '{my_title}'.")
            else:
                print(f"No relationships parsed for paragraph '{my_title}'; skipping Neo4j upload.")

Uploaded 9 relationships for paragraph 'Introduction'.
No relationships parsed for paragraph 'Stage 1: Data Preparation'; skipping Neo4j upload.
No relationships parsed for paragraph '4.1     Steps Involved in Model Initialisation'; skipping Neo4j upload.
Uploaded 12 relationships for paragraph 'Stage 3: Training Setup'.
No relationships parsed for paragraph '6.4.2    Comparison between HFT and LoRA'; skipping Neo4j upload.
Uploaded 5 relationships for paragraph 'Table 6.3: Comparative Analysis of Half Fine-Tuning (HFT) and Low-Rank Adaptation (LoRA).'.
Uploaded 4 relationships for paragraph '• Prompt'.
Uploaded 4 relationships for paragraph '• Supervised Fine-tuning Loss (SFT):'.
No relationships parsed for paragraph 'where yik is a binary indicator for the i-th token in the vocabulary, and pki is its predicted probability.'; skipping Neo4j upload.
Uploaded 4 relationships for paragraph 'Stage 5: Evaluation and Validation'.
No relationships parsed for paragraph 'Other tunable paramete

In [22]:
# Separate embeddings for section titles and paragraph titles + separate retrieval
from typing import List, Tuple, Optional
import json
import math
import ollama

def classify_title_kind(title: str) -> str:
    """Heuristic split between section titles and paragraph titles."""
    t = title.strip()
    if re.match(r"^(section\s+)?\d+(\.\d+)*([:)\-\s]|$)", t, flags=re.IGNORECASE):
        return "section"
    return "paragraph"

def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()

def split_titles_for_embedding(structured_data: List[dict]) -> Tuple[List[str], List[str]]:
    section_titles = []
    paragraph_titles = []
    for item in structured_data:
        if item.get("type") != "Title":
            continue
        title = normalize_text(item.get("text", ""))
        if not title:
            continue
        if classify_title_kind(title) == "section":
            section_titles.append(title)
        else:
            paragraph_titles.append(title)
    return paragraph_titles, section_titles

def safe_embed_text(
    model: str,
    text: str,
    initial_max_chars: int = 8000,
    min_chars: int = 16,
) -> Optional[List[float]]:
    """Embed text with adaptive truncation when model context is exceeded."""
    text = normalize_text(text)
    if not text:
        return None

    max_chars = min(initial_max_chars, len(text))
    last_error = None

    # Always attempt at least once, even for short strings.
    while True:
        candidate = text[:max_chars]
        try:
            return ollama.embeddings(model=model, prompt=candidate)["embedding"]
        except Exception as e:
            last_error = e
            msg = str(e).lower()
            if "input length exceeds the context length" in msg or "context length" in msg:
                if max_chars <= min_chars:
                    break
                next_max = max(min_chars, int(max_chars * 0.75))
                if next_max >= max_chars:
                    next_max = max_chars - 1
                max_chars = max(1, next_max)
                continue
            print(f"Warning: embedding failed for reason other than context length: {e}")
            return None

    print(f"Warning: could not embed text after adaptive truncation. text_prefix='{text[:80]}...', last_error={last_error}")
    return None

def build_embedding_records(titles: List[str], model: str) -> List[dict]:
    records = []
    failed = 0
    for title in titles:
        emb = safe_embed_text(model=model, text=title)
        if emb is None:
            failed += 1
            continue
        records.append({"name": title, "embedding": emb})

    if failed:
        print(f"Skipped {failed} titles that could not be embedded.")
    return records

def save_vector_store(path: str, records: List[dict]):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f)

def load_vector_store(path: str) -> List[dict]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def cosine_similarity(a: List[float], b: List[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

def retrieve_similar_titles(vector_store_path: str, query_text: str, model: str, top_k: int = 5):
    records = load_vector_store(vector_store_path)
    query_embedding = safe_embed_text(model=model, text=query_text)
    if query_embedding is None:
        print("Query embedding failed after adaptive truncation.")
        return []

    scored = []
    for row in records:
        score = cosine_similarity(query_embedding, row["embedding"])
        scored.append({"name": row["name"], "score": score})

    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:top_k]

# Build two separate vector storages
paragraph_titles, section_titles = split_titles_for_embedding(data)
print(f"Paragraph titles: {len(paragraph_titles)}")
print(f"Section titles: {len(section_titles)}")

paragraph_records = build_embedding_records(paragraph_titles, EMBEDDING_MODEL)
section_records = build_embedding_records(section_titles, EMBEDDING_MODEL)

save_vector_store(PARAGRAPH_VECTOR_STORE_PATH, paragraph_records)
save_vector_store(SECTION_VECTOR_STORE_PATH, section_records)

print(f"Saved paragraph-title embeddings: {len(paragraph_records)} -> {PARAGRAPH_VECTOR_STORE_PATH}")
print(f"Saved section-title embeddings: {len(section_records)} -> {SECTION_VECTOR_STORE_PATH}")

# Example retrievals (kept separate)
paragraph_hits = retrieve_similar_titles(
    PARAGRAPH_VECTOR_STORE_PATH,
    query_text="core network architecture",
    model=EMBEDDING_MODEL,
    top_k=5,
    )
section_hits = retrieve_similar_titles(
    SECTION_VECTOR_STORE_PATH,
    query_text="security procedures",
    model=EMBEDDING_MODEL,
    top_k=5,
    )

print("\nTop paragraph-title matches:")
for hit in paragraph_hits:
    print(f"- {hit['name']} (score={hit['score']:.4f})")

print("\nTop section-title matches:")
for hit in section_hits:
    print(f"- {hit['name']} (score={hit['score']:.4f})")

Paragraph titles: 45411
Section titles: 341
Saved paragraph-title embeddings: 45411 -> paragraph_title_vectors.json
Saved section-title embeddings: 341 -> section_title_vectors.json

Top paragraph-title matches:
- ### 16.10.2 Network Architecture (score=0.8484)
- ## 5.48 Subscription-based routing to a target core network (score=0.7762)
- ### 4.2.9 Network Analytics architecture (score=0.7639)
- #### 5.4.12.2 Core Network Assistance for PEIPS (score=0.7620)
- ## 5.9 Core network security (score=0.7499)

Top section-title matches:
- 6.5.2.4.2.1 General procedure (score=0.6909)
- 6.7.5.3.4.2 Procedure (score=0.6594)
- 8.3.6.1.1.4.2 Procedure (score=0.6569)
- 8.3.6.1.2.4.2 Procedure (score=0.6549)
- 8.3.2.1.4.2 Procedure (score=0.6548)


In [23]:
# Query -> retrieve only paragraph titles
def report_paragraph_titles(query_text: str, top_k: int = 5):
    hits = retrieve_similar_titles(
        PARAGRAPH_VECTOR_STORE_PATH,
        query_text=query_text,
        model=EMBEDDING_MODEL,
        top_k=top_k,
    )

    print(f"Query: {query_text}")
    print("Top paragraph-title matches:")
    if not hits:
        print("No paragraph titles found. Run the embedding build cell first.")
        return

    for i, hit in enumerate(hits, start=1):
        print(f"{i}. {hit['name']} (score={hit['score']:.4f})")

# Edit this query each time you want a new retrieval
user_query = " What are the core principles of O-RAN? "
report_paragraph_titles(user_query, top_k=10)

Query:  What are the core principles of O-RAN? 
Top paragraph-title matches:
1. The following key principles apply for support of Network Slicing in NG-RAN: (score=0.7133)
2. #### 16.3.2.1 CN-RAN interaction and internal RAN aspects (score=0.6672)
3. #### 5.32.5.1 General principles (score=0.6608)
4. ### 16.20.2 Principles (score=0.6606)
5. ### A.4.3.1 General principles (score=0.6569)
6. #### 6.3.7.0 General principles (score=0.6558)
7. ## A.3.1 General principles (score=0.6536)
8. In this clause, the general principles and requirements related to the realization of network slicing in the NG-RAN for NR connected to 5GC and for E-UTRA connected to 5GC are given. (score=0.6536)
9. As a basis for the operation of UE Positioning in NG-RAN, the following assumptions apply: (score=0.6528)
10. In this case, the following principles apply: (score=0.6492)


In [24]:
# Retrieve top-3 paragraph titles, then fetch directly connected neighbors in Neo4j
def _get_paragraph_neighbors(session, title: str, limit: int):
    # 1) Exact match (fast path)
    rows = list(session.run(
        """
        MATCH (p:Paragraph {name: $title})
        OPTIONAL MATCH (p)-[r]-(n)
        RETURN
        p.name AS paragraph_name,
        type(r) AS rel_type,
        labels(n) AS neighbor_labels,
        coalesce(n.name, n.title, toString(id(n))) AS neighbor_name
        LIMIT $limit
        """,
        title=title,
        limit=limit,
    ))

    if rows:
        return rows, "exact"

    # 2) Fallback: normalized/case-insensitive match for small formatting differences
    rows = list(session.run(
        """
        MATCH (p:Paragraph)
        WHERE toLower(trim(p.name)) = toLower(trim($title))
           OR toLower(trim(p.name)) CONTAINS toLower(trim($title))
           OR toLower(trim($title)) CONTAINS toLower(trim(p.name))
        OPTIONAL MATCH (p)-[r]-(n)
        RETURN
        p.name AS paragraph_name,
        type(r) AS rel_type,
        labels(n) AS neighbor_labels,
        coalesce(n.name, n.title, toString(id(n))) AS neighbor_name
        LIMIT $limit
        """,
        title=title,
        limit=limit,
    ))

    if rows:
        return rows, "fallback"

    return [], "none"

def report_top3_with_graph_neighbors(query_text: str, top_k: int = 3, neighbors_per_title: int = 20):
    top_hits = retrieve_similar_titles(
        PARAGRAPH_VECTOR_STORE_PATH,
        query_text=query_text,
        model=EMBEDDING_MODEL,
        top_k=top_k,
    )

    print(f"Query: {query_text}")
    if not top_hits:
        print("No paragraph-title hits found. Build vector stores first.")
        return

    print("Top paragraph-title hits:")
    for i, hit in enumerate(top_hits, start=1):
        print(f"{i}. {hit['name']} (score={hit['score']:.4f})")

    with GraphDatabase.driver(URI, auth=NEO4J_AUTH) as driver:
        driver.verify_connectivity()
        with driver.session() as session:
            for i, hit in enumerate(top_hits, start=1):
                title = hit["name"]
                print(f"\nNeighbors for result {i}: {title}")

                rows, match_mode = _get_paragraph_neighbors(session, title, neighbors_per_title)
                if match_mode == "fallback":
                    print("  Note: used fallback paragraph-name matching.")

                if not rows:
                    print("  Paragraph node not found in graph.")
                    continue

                has_neighbors = False
                for row in rows:
                    if row["rel_type"] is None or row["neighbor_name"] is None:
                        continue
                    has_neighbors = True
                    lbl = ":".join(row["neighbor_labels"]) if row["neighbor_labels"] else "Unknown"
                    print(f"  - [{row['rel_type']}] {row['neighbor_name']} ({lbl})")

                if not has_neighbors:
                    print("  Paragraph found, but no directly connected neighbors.")

# Edit query and run this cell
graph_query = user_query if 'user_query' in globals() else "security architecture in 5G core"
report_top3_with_graph_neighbors(graph_query, top_k=3, neighbors_per_title=25)

Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=8, column=44, offset=238>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 238, 'line': 8, 'column': 44}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (p:Paragraph {name: $title})\n        OPTIONAL MATCH (p)-[r]-(n)\n        RETURN\n        p.name AS paragraph_name,\n        type(r) AS rel_type,\n        labels(n) AS neighbor_labels,\n        coalesce(n.name, n.title, toString(id(n))) AS neighbor_name\n        LIMIT $limit\n        '
Received notification from DBMS server:

Query:  What are the core principles of O-RAN? 
Top paragraph-title hits:
1. The following key principles apply for support of Network Slicing in NG-RAN: (score=0.7133)
2. #### 16.3.2.1 CN-RAN interaction and internal RAN aspects (score=0.6672)
3. #### 5.32.5.1 General principles (score=0.6608)

Neighbors for result 1: The following key principles apply for support of Network Slicing in NG-RAN:


Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=11, column=44, offset=417>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 417, 'line': 11, 'column': 44}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (p:Paragraph)\n        WHERE toLower(trim(p.name)) = toLower(trim($title))\n           OR toLower(trim(p.name)) CONTAINS toLower(trim($title))\n           OR toLower(trim($title)) CONTAINS toLower(trim(p.name))\n        OPTIONAL MATCH (p)-[r]-(n)\n        RETURN\n        p.name AS paragraph_name,\n        type(r) AS rel_ty

  Paragraph node not found in graph.

Neighbors for result 2: #### 16.3.2.1 CN-RAN interaction and internal RAN aspects
  Paragraph node not found in graph.

Neighbors for result 3: #### 5.32.5.1 General principles
  Paragraph node not found in graph.


In [25]:
# For each of the top-3 paragraph titles, retrieve first 3 neighbors and map them to section chunks
def build_section_chunk_map(structured_data):
    section_chunks = {}
    current_section = None
    buffer_lines = []

    for item in structured_data:
        item_type = item.get("type")
        text = item.get("text", "").strip()
        if not text:
            continue

        if item_type == "Title" and classify_title_kind(text) == "section":
            if current_section is not None:
                section_chunks[current_section] = "\n".join(buffer_lines).strip()
            current_section = text
            buffer_lines = []
            continue

        if current_section is not None and item_type in {"Title", "Paragraph"}:
            buffer_lines.append(text)

    if current_section is not None:
        section_chunks[current_section] = "\n".join(buffer_lines).strip()

    return section_chunks

def _get_neighbor_names_for_paragraph(session, title: str, limit: int):
    # 1) Exact paragraph name
    rows = list(session.run(
        """
        MATCH (p:Paragraph {name: $title})-[r]-(n)
        RETURN coalesce(n.name, elementId(n)) AS neighbor_name
        LIMIT $limit
        """,
        title=title,
        limit=limit,
    ))
    if rows:
        return rows, "exact"

    # 2) Fallback paragraph match
    rows = list(session.run(
        """
        MATCH (p:Paragraph)-[r]-(n)
        WHERE toLower(trim(p.name)) = toLower(trim($title))
           OR toLower(trim(p.name)) CONTAINS toLower(trim($title))
           OR toLower(trim($title)) CONTAINS toLower(trim(p.name))
        RETURN coalesce(n.name, elementId(n)) AS neighbor_name
        LIMIT $limit
        """,
        title=title,
        limit=limit,
    ))
    if rows:
        return rows, "fallback"

    return [], "none"

def neighbors_per_paragraph_hit(query_text: str, paragraph_top_k: int = 3, neighbors_per_title: int = 3):
    paragraph_hits = retrieve_similar_titles(
        PARAGRAPH_VECTOR_STORE_PATH,
        query_text=query_text,
        model=EMBEDDING_MODEL,
        top_k=paragraph_top_k,
    )

    grouped = []
    with GraphDatabase.driver(URI, auth=NEO4J_AUTH) as driver:
        driver.verify_connectivity()
        with driver.session() as session:
            for hit in paragraph_hits:
                rows, match_mode = _get_neighbor_names_for_paragraph(
                    session,
                    title=hit["name"],
                    limit=neighbors_per_title,
                )

                neighbors = []
                seen = set()
                for row in rows:
                    name = row["neighbor_name"]
                    if not name or name in seen:
                        continue
                    seen.add(name)
                    neighbors.append(name)

                grouped.append({
                    "paragraph_title": hit["name"],
                    "paragraph_score": hit["score"],
                    "neighbors": neighbors,
                    "match_mode": match_mode,
                })

    return grouped

def neighbor_to_section_chunks(query_text: str, paragraph_top_k: int = 3, neighbors_per_title: int = 3, section_top_k: int = 1):
    divider_main = "=" * 100
    divider_sub = "-" * 100
    divider_chunk = "." * 100

    section_chunk_map = build_section_chunk_map(data)
    grouped_hits = neighbors_per_paragraph_hit(
        query_text=query_text,
        paragraph_top_k=paragraph_top_k,
        neighbors_per_title=neighbors_per_title,
    )

    print(divider_main)
    print("PHASE 1 - INPUT QUERY")
    print(divider_main)
    print(f"Query: {query_text}")

    if not grouped_hits:
        print(divider_sub)
        print("No paragraph hits found.")
        return

    print("\n" + divider_main)
    print("PHASE 2 - TOP PARAGRAPH TITLES + FIRST 3 NEIGHBORS EACH")
    print(divider_main)
    for i, g in enumerate(grouped_hits, start=1):
        mode_note = "" if g.get("match_mode") != "fallback" else " [fallback match]"
        print(f"{i}. {g['paragraph_title']} (score={g['paragraph_score']:.4f}){mode_note}")
        if g["neighbors"]:
            for j, n in enumerate(g["neighbors"], start=1):
                print(f"   {j}) {n}")
        else:
            print("   No neighbors found for this title.")
        print(divider_sub)

    print("\n" + divider_main)
    print("PHASE 3 - SECTION RETRIEVAL + FULL CHUNK FOR EACH NEIGHBOR")
    print(divider_main)
    for i, g in enumerate(grouped_hits, start=1):
        print(f"\nParagraph {i}: {g['paragraph_title']}")
        print(divider_sub)
        if not g["neighbors"]:
            print("Skipped: no neighbors.")
            continue

        for neighbor in g["neighbors"]:
            print(f"Neighbor query: {neighbor}")
            section_hits = retrieve_similar_titles(
                SECTION_VECTOR_STORE_PATH,
                query_text=neighbor,
                model=EMBEDDING_MODEL,
                top_k=section_top_k,
            )

            if not section_hits:
                print("  No section hits found.")
                print(divider_chunk)
                continue

            for hit in section_hits:
                section_title = hit["name"]
                chunk = section_chunk_map.get(section_title, "")
                print(f"  Matched section: {section_title} (score={hit['score']:.4f})")
                if chunk:
                    print("  Chunk:")
                    print(chunk)
                else:
                    print("  Chunk not found in parsed data.")
                print(divider_chunk)

# Edit query and run
neighbor_section_query = graph_query if 'graph_query' in globals() else "LLM Fine tuning"
neighbor_to_section_chunks(
    query_text=neighbor_section_query,
    paragraph_top_k=3,
    neighbors_per_title=3,
    section_top_k=1,
    )

PHASE 1 - INPUT QUERY
Query:  What are the core principles of O-RAN? 

PHASE 2 - TOP PARAGRAPH TITLES + FIRST 3 NEIGHBORS EACH
1. The following key principles apply for support of Network Slicing in NG-RAN: (score=0.7133)
   No neighbors found for this title.
----------------------------------------------------------------------------------------------------
2. #### 16.3.2.1 CN-RAN interaction and internal RAN aspects (score=0.6672)
   No neighbors found for this title.
----------------------------------------------------------------------------------------------------
3. #### 5.32.5.1 General principles (score=0.6608)
   No neighbors found for this title.
----------------------------------------------------------------------------------------------------

PHASE 3 - SECTION RETRIEVAL + FULL CHUNK FOR EACH NEIGHBOR

Paragraph 1: The following key principles apply for support of Network Slicing in NG-RAN:
-----------------------------------------------------------------------------------

In [21]:
# Diagnostic: verify embedding model availability and a minimal embedding call
try:
    models_resp = ollama.list()
    model_names = []
    for m in models_resp.get("models", []):
        name = m.get("model") or m.get("name")
        if name:
            model_names.append(name)

    print("Loaded models:")
    for name in model_names[:20]:
        print("-", name)

    print(f"\nCurrent EMBEDDING_MODEL: {EMBEDDING_MODEL}")
    has_model = any(name.startswith(EMBEDDING_MODEL) or EMBEDDING_MODEL in name for name in model_names)
    print("Model found locally:", has_model)

    test_prompt = "hello world"
    print(f"\nTesting embeddings with prompt: '{test_prompt}'")
    out = ollama.embeddings(model=EMBEDDING_MODEL, prompt=test_prompt)
    emb = out.get("embedding")
    print("Embedding success:", emb is not None)
    if emb is not None:
        print("Embedding length:", len(emb))
except Exception as e:
    print("Embedding diagnostic failed:")
    print(type(e).__name__, str(e))

Loaded models:
- qwen3.5:latest
- sciphi/triplex:latest
- mxbai-embed-large:latest
- deepseek-r1:1.5b
- all-minilm:latest
- hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF:latest
- llama3:8b
- ORANSight_Qwen_1.5B_Instruct:latest
- phi:latest
- tinyllama:latest
- hf.co/CompendiumLabs/bge-base-en-v1.5-gguf:latest

Current EMBEDDING_MODEL: mxbai-embed-large
Model found locally: True

Testing embeddings with prompt: 'hello world'
Embedding success: True
Embedding length: 1024


In [33]:
# Self-contained RAG generation cell with MCQ option scoring
from typing import Dict, Any, List, Tuple, Optional
import json
import math
import re
import ollama


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def canonical_title_key(title: str) -> str:
    t = re.sub(r"^#+\s*", "", (title or "").strip())
    return normalize_text(t).lower()


def resolve_embedding_model(preferred_model: str) -> Optional[str]:
    try:
        models_resp = ollama.list()
    except Exception as e:
        print(f"Could not list Ollama models: {e}")
        return None

    model_names = []
    for m in models_resp.get("models", []):
        name = m.get("model") or m.get("name")
        if name:
            model_names.append(name)

    if not model_names:
        return None

    if any(name == preferred_model or name.startswith(f"{preferred_model}:") for name in model_names):
        return preferred_model

    embedding_candidates = [name for name in model_names if "embed" in name.lower()]
    if embedding_candidates:
        print(f"Using fallback embedding model: {embedding_candidates[0]}")
        return embedding_candidates[0]

    print("No embedding model found locally. Install one (e.g., `ollama pull nomic-embed-text`).")
    return None


def load_vector_store(path: str) -> List[dict]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def cosine_similarity(a: List[float], b: List[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


def safe_embed_text(
    model: str,
    text: str,
    initial_max_chars: int = 8000,
    min_chars: int = 16,
) -> Optional[List[float]]:
    text = normalize_text(text)
    if not text:
        return None

    max_chars = min(initial_max_chars, len(text))
    while True:
        candidate = text[:max_chars]
        try:
            return ollama.embeddings(model=model, prompt=candidate)["embedding"]
        except Exception as e:
            msg = str(e).lower()
            if "context length" in msg and max_chars > min_chars:
                next_max = max(min_chars, int(max_chars * 0.75))
                if next_max >= max_chars:
                    next_max = max_chars - 1
                max_chars = max(1, next_max)
                continue
            print(f"Embedding failed: {e}")
            return None


def retrieve_similar_titles_local(vector_store_path: str, query_text: str, model: str, top_k: int = 5) -> List[dict]:
    records = load_vector_store(vector_store_path)
    query_embedding = safe_embed_text(model=model, text=query_text)
    if query_embedding is None:
        return []

    scored = []
    for row in records:
        score = cosine_similarity(query_embedding, row["embedding"])
        scored.append({"name": row["name"], "score": score})

    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:top_k]


def build_section_chunk_map_local(structured_data: List[dict]) -> Dict[str, str]:
    section_chunks: Dict[str, str] = {}
    current_section = None
    buffer_lines: List[str] = []

    def _store_section(section_title: str, lines: List[str]):
        if not section_title:
            return
        chunk = "\n".join(lines).strip()
        if not chunk:
            chunk = section_title.strip()
        if not chunk:
            return
        raw_key = section_title.strip()
        canon_key = canonical_title_key(section_title)
        section_chunks[raw_key] = chunk
        section_chunks[canon_key] = chunk

    for item in structured_data:
        item_type = item.get("type")
        text = item.get("text", "").strip()
        if not text:
            continue

        if item_type == "Title":
            if current_section is not None:
                _store_section(current_section, buffer_lines)
            current_section = text
            buffer_lines = []
            continue

        if current_section is not None and item_type == "Paragraph":
            buffer_lines.append(text)

    if current_section is not None:
        _store_section(current_section, buffer_lines)

    return section_chunks


def get_neighbor_names_for_paragraph_local(session, title: str, limit: int = 3) -> Tuple[List[str], str]:
    rows = list(session.run(
        """
        MATCH (p:Paragraph {name: $title})-[r]-(n)
        RETURN coalesce(n.name, elementId(n)) AS neighbor_name
        LIMIT $limit
        """,
        title=title,
        limit=limit,
    ))
    if rows:
        return [r["neighbor_name"] for r in rows if r["neighbor_name"]], "exact"

    rows = list(session.run(
        """
        MATCH (p:Paragraph)-[r]-(n)
        WHERE toLower(trim(p.name)) = toLower(trim($title))
           OR toLower(trim(p.name)) CONTAINS toLower(trim($title))
           OR toLower(trim($title)) CONTAINS toLower(trim(p.name))
        RETURN coalesce(n.name, elementId(n)) AS neighbor_name
        LIMIT $limit
        """,
        title=title,
        limit=limit,
    ))
    if rows:
        return [r["neighbor_name"] for r in rows if r["neighbor_name"]], "fallback"

    return [], "none"


def _lookup_chunk(section_chunk_map: Dict[str, str], section_title: str) -> str:
    return section_chunk_map.get(section_title, "") or section_chunk_map.get(canonical_title_key(section_title), "")


def _query_terms(query_text: str) -> set:
    terms = set(re.findall(r"[a-zA-Z0-9]{3,}", query_text.lower()))
    return {t for t in terms if t not in {"what", "which", "with", "from", "that", "this", "return", "exactly", "line", "format"}}


def _query_requires_oran(query_text: str) -> bool:
    q = query_text.lower()
    return "o-ran" in q or "oran" in q


def _is_relevant_chunk(query_text: str, chunk_text: str, min_overlap: int = 1) -> bool:
    chunk_low = (chunk_text or "").lower()
    if _query_requires_oran(query_text) and ("o-ran" not in chunk_low and "oran" not in chunk_low):
        return False

    q_terms = _query_terms(query_text)
    if not q_terms:
        return True
    c_terms = set(re.findall(r"[a-zA-Z0-9]{3,}", chunk_low))
    return len(q_terms & c_terms) >= min_overlap


def collect_retrieved_context(
    query_text: str,
    embedding_model: str,
    paragraph_top_k: int = 5,
    neighbors_per_title: int = 5,
    section_top_k: int = 3,
    max_chunks: int = 8,
) -> Dict[str, Any]:
    section_chunk_map = build_section_chunk_map_local(data)

    paragraph_hits = retrieve_similar_titles_local(
        PARAGRAPH_VECTOR_STORE_PATH,
        query_text=query_text,
        model=embedding_model,
        top_k=paragraph_top_k,
    )

    grouped_hits: List[Dict[str, Any]] = []
    contexts: List[Dict[str, Any]] = []
    seen_context = set()

    with GraphDatabase.driver(URI, auth=NEO4J_AUTH) as driver:
        driver.verify_connectivity()
        with driver.session() as session:
            for hit in paragraph_hits:
                neighbors, match_mode = get_neighbor_names_for_paragraph_local(
                    session=session,
                    title=hit["name"],
                    limit=neighbors_per_title,
                )
                grouped_hits.append({
                    "paragraph_title": hit["name"],
                    "paragraph_score": hit["score"],
                    "neighbors": neighbors,
                    "match_mode": match_mode,
                })

                for neighbor in neighbors:
                    section_hits = retrieve_similar_titles_local(
                        SECTION_VECTOR_STORE_PATH,
                        query_text=neighbor,
                        model=embedding_model,
                        top_k=section_top_k,
                    )
                    for sec_hit in section_hits:
                        section_title = sec_hit["name"]
                        chunk_text = _lookup_chunk(section_chunk_map, section_title).strip()
                        if not chunk_text or not _is_relevant_chunk(query_text, chunk_text, min_overlap=1):
                            continue

                        unique_key = (canonical_title_key(section_title), chunk_text[:200])
                        if unique_key in seen_context:
                            continue
                        seen_context.add(unique_key)

                        contexts.append({
                            "paragraph_title": hit["name"],
                            "paragraph_score": hit["score"],
                            "neighbor": neighbor,
                            "section_title": section_title,
                            "section_score": sec_hit["score"],
                            "chunk": chunk_text,
                            "retrieval_mode": "graph_neighbor_to_section",
                        })

                        if len(contexts) >= max_chunks:
                            break
                    if len(contexts) >= max_chunks:
                        break
                if len(contexts) >= max_chunks:
                    break

    if not contexts:
        direct_hits = retrieve_similar_titles_local(
            SECTION_VECTOR_STORE_PATH,
            query_text=query_text,
            model=embedding_model,
            top_k=max(section_top_k, min(max_chunks * 2, 12)),
        )
        for sec_hit in direct_hits:
            section_title = sec_hit["name"]
            chunk_text = _lookup_chunk(section_chunk_map, section_title).strip()
            if not chunk_text or not _is_relevant_chunk(query_text, chunk_text, min_overlap=1):
                continue
            contexts.append({
                "paragraph_title": "N/A",
                "paragraph_score": 0.0,
                "neighbor": "DIRECT_QUERY",
                "section_title": section_title,
                "section_score": sec_hit["score"],
                "chunk": chunk_text,
                "retrieval_mode": "direct_section_fallback",
            })
            if len(contexts) >= max_chunks:
                break

    return {
        "query": query_text,
        "grouped_hits": grouped_hits,
        "contexts": contexts,
    }


def parse_mcq_options(query_text: str) -> List[Tuple[str, str]]:
    opts: List[Tuple[str, str]] = []
    for line in query_text.splitlines():
        m = re.match(r"^\s*([A-D])\)\s*(.+?)\s*$", line)
        if m:
            opts.append((m.group(1), m.group(2)))
    return opts


def score_mcq_options(query_text: str, options: List[Tuple[str, str]], contexts: List[Dict[str, Any]], embedding_model: str) -> List[dict]:
    query_emb = safe_embed_text(embedding_model, query_text)
    if query_emb is None:
        return []

    chunk_embs = []
    for ctx in contexts:
        emb = safe_embed_text(embedding_model, ctx.get("chunk", ""))
        if emb is not None:
            chunk_embs.append(emb)

    if not chunk_embs:
        return []

    scored = []
    for letter, text in options:
        opt_emb = safe_embed_text(embedding_model, text)
        if opt_emb is None:
            continue
        q_sim = cosine_similarity(query_emb, opt_emb)
        c_sim = max(cosine_similarity(opt_emb, ce) for ce in chunk_embs)
        total = 0.35 * q_sim + 0.65 * c_sim
        scored.append({"letter": letter, "text": text, "score": total, "query_similarity": q_sim, "context_similarity": c_sim})

    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored


def llm_pick_mcq_option(query_text: str, llm_model: str) -> str:
    prompt = (
        "You are answering a telecom multiple-choice question. "
        "Use your domain knowledge. Return exactly one line in format: <LETTER>) <OPTION_TEXT>.\n\n"
        f"Question:\n{query_text}"
    )
    response = ollama.chat(model=llm_model, messages=[{"role": "user", "content": prompt}])
    answer = response["message"]["content"].strip().splitlines()[0]
    return answer


def build_rag_prompt(query_text: str, contexts: List[Dict[str, Any]], output_mode: str = "full") -> str:
    blocks = []
    for i, ctx in enumerate(contexts, start=1):
        blocks.append(
            f"Context {i}\n"
            f"- Retrieval mode: {ctx.get('retrieval_mode', 'unknown')}\n"
            f"- Section title: {ctx['section_title']}\n"
            f"- Section similarity: {ctx['section_score']:.4f}\n"
            f"- Chunk:\n{ctx['chunk']}"
        )
    context_text = "\n\n".join(blocks)

    if output_mode == "one_line":
        return (
            "Answer ONLY from context. "
            "If unsure, say: Insufficient context.\n\n"
            f"Question:\n{query_text}\n\n"
            f"Context:\n{context_text}\n\n"
            "Output exactly one line."
        )

    return (
        "You are a technical assistant for O-RAN and 3GPP documents. "
        "Answer ONLY using the provided context. "
        "If the context does not contain the answer, explicitly say: 'Insufficient context.'\n\n"
        f"User question:\n{query_text}\n\n"
        f"Retrieved context:\n{context_text}\n\n"
        "Return:\n"
        "1) A concise answer\n"
        "2) Bullet list of supporting context references (section titles)\n"
        "3) Confidence: High/Medium/Low"
    )


def generate_answer_from_retrieved_data(
    query_text: str,
    llm_model: str = "llama3:8b",
    paragraph_top_k: int = 5,
    neighbors_per_title: int = 5,
    section_top_k: int = 3,
    max_chunks: int = 6,
    output_mode: str = "full",
    use_mcq_option_scoring: bool = True,
) -> Dict[str, Any]:
    embedding_model = resolve_embedding_model(EMBEDDING_MODEL)
    options = parse_mcq_options(query_text)

    if embedding_model is None:
        fallback_answer = llm_pick_mcq_option(query_text, llm_model) if options else "Insufficient context: no local embedding model available."
        return {
            "query": query_text,
            "model": llm_model,
            "embedding_model": None,
            "answer": fallback_answer,
            "contexts": [],
            "grouped_hits": [],
            "option_scores": [],
        }

    retrieval = collect_retrieved_context(
        query_text=query_text,
        embedding_model=embedding_model,
        paragraph_top_k=paragraph_top_k,
        neighbors_per_title=neighbors_per_title,
        section_top_k=section_top_k,
        max_chunks=max_chunks,
    )

    if use_mcq_option_scoring and options:
        # If retrieval is weak/empty for MCQ, fall back to direct option selection via LLM.
        if not retrieval["contexts"]:
            fallback_answer = llm_pick_mcq_option(query_text, llm_model)
            return {
                "query": query_text,
                "model": llm_model,
                "embedding_model": embedding_model,
                "answer": fallback_answer,
                "contexts": [],
                "grouped_hits": retrieval["grouped_hits"],
                "option_scores": [],
            }

        option_scores = score_mcq_options(query_text, options, retrieval["contexts"], embedding_model)
        if option_scores:
            best = option_scores[0]
            return {
                "query": query_text,
                "model": llm_model,
                "embedding_model": embedding_model,
                "answer": f"{best['letter']}) {best['text']}",
                "contexts": retrieval["contexts"],
                "grouped_hits": retrieval["grouped_hits"],
                "option_scores": option_scores,
            }

    if not retrieval["contexts"]:
        return {
            "query": query_text,
            "model": llm_model,
            "embedding_model": embedding_model,
            "answer": "Insufficient context: retrieval returned no relevant chunks.",
            "contexts": [],
            "grouped_hits": retrieval["grouped_hits"],
            "option_scores": [],
        }

    prompt = build_rag_prompt(query_text, retrieval["contexts"], output_mode=output_mode)
    response = ollama.chat(model=llm_model, messages=[{"role": "user", "content": prompt}])
    answer = response["message"]["content"]

    return {
        "query": query_text,
        "model": llm_model,
        "embedding_model": embedding_model,
        "answer": answer,
        "contexts": retrieval["contexts"],
        "grouped_hits": retrieval["grouped_hits"],
        "option_scores": [],
    }


# Run end-to-end sanity check
generation_query = "What are the core principles of O-RAN?"
rag_result = generate_answer_from_retrieved_data(
    query_text=generation_query,
    llm_model="llama3:8b",
    paragraph_top_k=5,
    neighbors_per_title=5,
    section_top_k=3,
    max_chunks=6,
    output_mode="full",
)

print("Query:", rag_result["query"])
print("Model:", rag_result["model"])
print("Embedding model:", rag_result.get("embedding_model"))
print("Retrieved contexts:", len(rag_result["contexts"]))
print("\nGenerated answer:\n")
print(rag_result["answer"])

Using fallback embedding model: mxbai-embed-large:latest
Query: What are the core principles of O-RAN?
Model: llama3:8b
Embedding model: mxbai-embed-large:latest
Retrieved contexts: 0

Generated answer:

Insufficient context: retrieval returned no relevant chunks.


In [41]:
! ollama list

]11;?\NAME                                                 ID              SIZE      MODIFIED           
mistral-nemo:latest                                  e7e06d107c6c    7.1 GB    About a minute ago    
qwen3.5:latest                                       6488c96fa5fa    6.6 GB    4 days ago            
sciphi/triplex:latest                                6071d91b5626    2.4 GB    7 days ago            
mxbai-embed-large:latest                             468836162de7    669 MB    10 days ago           
deepseek-r1:1.5b                                     e0979632db5a    1.1 GB    11 days ago           
all-minilm:latest                                    1b226e2802db    45 MB     11 days ago           
hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF:latest    042cf58aa32f    807 MB    3 weeks ago           
llama3:8b                                            365c0bd3c000    4.7 GB    3 weeks ago           
ORANSight_Qwen_1.5B_Instruct:latest                  07bc24db663e    3.1 GB  

In [66]:
# MCQ precision test with option scoring
mcq_query = """Which O-RAN Working Group focuses on the architecture description of Open Radio Access Networks?

A) O-RAN.WG3

B)O-RAN.WG4

C)  O-RAN.WG1

D) O-RAN.WG5

Return exactly one line in this format: <LETTER>) <OPTION_TEXT>"""

mcq_result = generate_answer_from_retrieved_data(
    query_text=mcq_query,
    llm_model="mistral-nemo",
    paragraph_top_k=8,
    neighbors_per_title=8,
    section_top_k=5,
    max_chunks=8,
    output_mode="one_line",
    use_mcq_option_scoring=True,
)

print("Retrieved contexts:", len(mcq_result["contexts"]))
print("Model answer:", mcq_result["answer"])

if mcq_result.get("option_scores"):
    print("\nOption scores:")
    for row in mcq_result["option_scores"]:
        print(f"{row['letter']}) {row['text']} -> total={row['score']:.4f} (q={row['query_similarity']:.4f}, c={row['context_similarity']:.4f})")

Using fallback embedding model: mxbai-embed-large:latest
Retrieved contexts: 0
Model answer: B) O-RAN.WG4


In [28]:
# Diagnostic: check embedding dimension consistency between stored vectors and active model
active_model = resolve_embedding_model(EMBEDDING_MODEL)
section_records = load_vector_store(SECTION_VECTOR_STORE_PATH)
stored_dim = len(section_records[0]["embedding"]) if section_records else 0

probe_emb = safe_embed_text(active_model, "diagnostic query") if active_model else None
active_dim = len(probe_emb) if probe_emb else 0

print("Active embedding model:", active_model)
print("Stored vector dimension:", stored_dim)
print("Active model embedding dimension:", active_dim)
print("Dimension match:", stored_dim == active_dim)

Using fallback embedding model: mxbai-embed-large:latest
Active embedding model: mxbai-embed-large:latest
Stored vector dimension: 1024
Active model embedding dimension: 1024
Dimension match: True


In [30]:
# Diagnostic: inspect retrieved context metadata for the last MCQ run
if "mcq_result" in globals() and mcq_result.get("contexts"):
    for i, ctx in enumerate(mcq_result["contexts"], start=1):
        print(f"{i}. mode={ctx.get('retrieval_mode')} section={ctx.get('section_title')} score={ctx.get('section_score'):.4f}")
        preview = (ctx.get("chunk") or "")[:180].replace("\n", " ")
        print("   chunk preview:", preview)
else:
    print("No mcq_result contexts available.")

1. mode=direct_section_fallback section=9.5.3.3.4.2 Requirements for network signalling values “NS_205N” and “NS_206N” score=0.6092
   chunk preview: 9.5.3.3.4.2	Requirements for network signalling values “NS_205N” and “NS_206N”
